<a href="https://colab.research.google.com/github/ddickson28/Captain-FPSO-BN/blob/Change-to-cumulative-damage/CumulativeDamage29_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%pip install mbnpy

Note: you may need to restart the kernel to use updated packages.


c:\Users\Lindsay\AppData\Local\spyder-6\python.exe: No module named pip


In [9]:
#Import modules

from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import numpy as np

ModuleNotFoundError: No module named 'mbnpy'

In [ ]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference
import numpy as np

# --- Parameters ---
n_components = 2  # geographical locations
n_timesteps  = 2  # time steps

# --- State spaces ---
states_damage = ['0','0.2','0.4','0.6','0.8','1.0']
states_bool   = ['False','True']
states_resist = ['0.4','0.6','0.8','1.0','1.2','1.4']
states_aux    = ['-0.5','-0.25','0.0','0.25','0.5']

# ──────────────────────────────────────────────
# 1. VARIABLES
# ──────────────────────────────────────────────
varis = {}

for loc in range(1, n_components + 1):
    for t in range(1, n_timesteps + 1):
        varis[f'Wx{loc}{t}']  = variable.Variable(f'Wx{loc}{t}',  states_damage)
        varis[f'Lx{loc}{t}']  = variable.Variable(f'Lx{loc}{t}',  states_damage)
        varis[f'Tx{loc}{t}']  = variable.Variable(f'Tx{loc}{t}',  states_damage)
        varis[f'Px{loc}{t}']  = variable.Variable(f'Px{loc}{t}',  states_bool)
        varis[f'Cx{loc}{t}']  = variable.Variable(f'Cx{loc}{t}',  states_damage)
        varis[f'Clx{loc}{t}'] = variable.Variable(f'CLx{loc}{t}', states_bool)
        varis[f'Rx{loc}{t}']  = variable.Variable(f'Rx{loc}{t}',  states_resist)
        varis[f'Zx{loc}{t}']  = variable.Variable(f'Zx{loc}{t}',  states_aux)
        varis[f'Vx{loc}{t}']  = variable.Variable(f'Vx{loc}{t}',  states_aux)
 
# Initial cumulative damage at t=0 per location (prior: no damage)
for loc in range(1, n_components + 1):
    varis[f'Cx{loc}0'] = variable.Variable(f'Cx{loc}0', states_damage)

# Ux{t}: one common spatial RV per time step.
# Each Ux{t} connects all n_components locations at time t but does not chain through time.
# Using a single Ux shared across all time steps caused a MemoryError: eliminating it
# required multiplying all n_components*n_timesteps Zx CPMs together, producing an
# intermediate factor with ~5^(2*n_components*n_timesteps + 1) rows (~49M for 3x3).
# With Ux{t}, elimination only multiplies n_components Zx CPMs → 5^4 = 625 rows.
for t in range(1, n_timesteps + 1):
    varis[f'Ux{t}'] = variable.Variable(f'Ux{t}', states_aux)

# ──────────────────────────────────────────────
# 2. CPM MATRICES  (defined once, reused for every loc/t)
# ──────────────────────────────────────────────

C_Tx = np.array([
    [0,0,0],[1,1,0],[2,2,0],[3,3,0],[4,4,0],[5,5,0],
    [1,0,1],[2,1,1],[3,2,1],[4,3,1],[5,4,1],[5,5,1],
    [0,0,2],[3,1,2],[4,2,2],[5,3,2],[5,4,2],[5,5,2],
    [3,0,3],[4,1,3],[5,2,3],[5,3,3],[5,4,3],[5,5,3],
    [4,0,4],[5,1,4],[5,2,4],[5,3,4],[5,4,4],[5,5,4],
    [5,0,5],[2,1,5],[5,2,5],[5,3,5],[5,4,5],[5,5,5],
], dtype=int)

# Cx{loc}{t} | Px{loc}{t}, Cx{loc}{t-1}, Tx{loc}{t}
C_Cx = np.array([
    [0,0,0,0],[0,1,0,0],[1,0,1,0],[1,1,1,0],[2,0,2,0],[0,1,2,0],
    [3,0,3,0],[0,1,3,0],[4,0,4,0],[0,1,4,0],[5,0,5,0],[0,1,5,0],
    [1,0,0,1],[1,1,0,1],[2,0,1,1],[1,1,1,1],[3,0,2,1],[1,1,2,1],
    [4,0,3,1],[1,1,3,1],[5,0,4,1],[1,1,4,1],[5,0,5,1],[1,1,5,1],
    [2,0,0,2],[2,1,0,2],[3,0,1,2],[2,1,1,2],[4,0,2,2],[2,1,2,2],
    [5,0,3,2],[2,1,3,2],[5,0,4,2],[2,1,4,2],[5,0,5,2],[2,1,5,2],
    [3,0,0,3],[3,1,0,3],[4,0,1,3],[3,1,1,3],[5,0,2,3],[3,1,2,3],
    [5,0,3,3],[3,1,3,3],[5,0,4,3],[3,1,4,3],[5,0,5,3],[3,1,5,3],
    [4,0,0,4],[4,1,0,4],[5,0,1,4],[4,1,1,4],[5,0,2,4],[4,1,2,4],
    [5,0,3,4],[4,1,3,4],[5,0,4,4],[4,1,4,4],[5,0,5,4],[4,1,5,4],
    [5,0,0,5],[5,1,0,5],[5,0,1,5],[5,1,1,5],[5,0,2,5],[5,1,2,5],
    [5,0,3,5],[5,1,3,5],[5,0,4,5],[5,1,4,5],[5,0,5,5],[5,1,5,5],
], dtype=int)

C_Rx = np.array([
    [1,0],[1,1],[2,2],[3,3],[4,4],
], dtype=int)

C_Clx = np.array([
    [0,0,0],[0,1,0],[1,2,0],[1,3,0],[1,4,0],[1,5,0],
    [0,0,1],[0,1,1],[0,2,1],[1,3,1],[1,4,1],[1,5,1],
    [0,0,2],[0,1,2],[0,2,2],[0,3,2],[1,4,2],[1,5,2],
    [0,0,3],[0,1,3],[0,2,3],[0,3,3],[0,4,3],[1,5,3],
    [0,0,4],[0,1,4],[0,2,4],[0,3,4],[0,4,4],[0,5,4],
    [0,0,5],[0,1,5],[0,2,5],[0,3,5],[0,4,5],[0,5,5],
], dtype=int)

# Zx{loc}{t} | Vx{loc}{t}, Ux{t}
C_Zx = np.array([
    [0,0,0],[0,1,0],[1,2,0],[2,3,0],[2,4,0],
    [0,0,1],[1,1,1],[2,2,1],[2,3,1],[3,4,1],
    [1,0,2],[2,1,2],[2,2,2],[3,3,2],[4,4,2],
    [2,0,3],[2,1,3],[3,2,3],[4,3,3],[4,4,3],
    [3,0,4],[3,1,4],[4,2,4],[4,3,4],[4,4,4],
], dtype=int)

# ──────────────────────────────────────────────
# 3. CPMs
# ──────────────────────────────────────────────
cpms = {}

for loc in range(1, n_components + 1):
    for t in range(1, n_timesteps + 1):
        cpms[f'Wx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Wx{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
            p=np.array([0.017, 0.435, 0.518, 0.03, 0.0, 0.0])
        )
        cpms[f'Lx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Lx{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
            p=np.array([0.122, 0.677, 0.198, 0.002, 0.0, 0.0])
        )
        cpms[f'Tx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Tx{loc}{t}'], varis[f'Wx{loc}{t}'], varis[f'Lx{loc}{t}']],
            no_child=1,
            C=C_Tx,
            p=np.ones(len(C_Tx))
        )
        cpms[f'Px{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Px{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1]], dtype=int),
            p=np.array([1.0, 0.0])
        )
        cpms[f'Vx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Vx{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1],[2],[3],[4]], dtype=int),
            p=np.array([0.0, 0.006, 0.493, 0.493, 0.006])
        )
        # Zx{loc}{t} depends on location-specific Vx and the time-step-specific Ux{t}
        cpms[f'Zx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Zx{loc}{t}'], varis[f'Vx{loc}{t}'], varis[f'Ux{t}']],
            no_child=1,
            C=C_Zx,
            p=np.ones(len(C_Zx))
        )
        cpms[f'Rx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Rx{loc}{t}'], varis[f'Zx{loc}{t}']],
            no_child=1,
            C=C_Rx,
            p=np.ones(len(C_Rx))
        )
        # Cx{loc}{t} chains temporally: parent is Cx{loc}{t-1} (same location, previous step)
        cpms[f'Cx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Cx{loc}{t}'], varis[f'Px{loc}{t}'], varis[f'Cx{loc}{t-1}'], varis[f'Tx{loc}{t}']],
            no_child=1,
            C=C_Cx,
            p=np.ones(len(C_Cx))
        )
        cpms[f'Clx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Clx{loc}{t}'], varis[f'Cx{loc}{t}'], varis[f'Rx{loc}{t}']],
            no_child=1,
            C=C_Clx,
            p=np.ones(len(C_Clx))
        )

# Initial cumulative damage: no damage at t=0
for loc in range(1, n_components + 1):
    cpms[f'Cx{loc}0'] = cpm.Cpm(
        variables=[varis[f'Cx{loc}0']],
        no_child=1,
        C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
        p=np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])
    )

# Ux{t}: one per time step, same prior as before
for t in range(1, n_timesteps + 1):
    cpms[f'Ux{t}'] = cpm.Cpm(
        variables=[varis[f'Ux{t}']],
        no_child=1,
        C=np.array([[0],[1],[2],[3],[4]], dtype=int),
        p=np.array([0.0, 0.006, 0.493, 0.493, 0.006])
    )

# ──────────────────────────────────────────────
# 4. ELIMINATION ORDER
# Query: Clx{n_components}{n_timesteps}  (last location, last time step)
#
# Order within each time step:
#   Wx, Lx, Tx → Px → Vx → Ux{t} → Zx → Rx → Clx (non-query)
# Eliminating Vx before Ux{t} shrinks the Zx CPM scope from
# {Zx, Vx, Ux{t}} to {Zx, Ux{t}}, so the product at Ux{t} elimination
# spans only n_components+1 variables (625 rows for n_components=3).
# Cx initial states and the temporal chain are eliminated last.
# ──────────────────────────────────────────────
varis_elim_order = []

for t in range(1, n_timesteps + 1):
    for loc in range(1, n_components + 1):
        varis_elim_order += [varis[f'Wx{loc}{t}'], varis[f'Lx{loc}{t}'], varis[f'Tx{loc}{t}']]
    for loc in range(1, n_components + 1):
        varis_elim_order.append(varis[f'Px{loc}{t}'])
    for loc in range(1, n_components + 1):
        varis_elim_order.append(varis[f'Vx{loc}{t}'])
    varis_elim_order.append(varis[f'Ux{t}'])
    for loc in range(1, n_components + 1):
        varis_elim_order.append(varis[f'Zx{loc}{t}'])
    for loc in range(1, n_components + 1):
        varis_elim_order.append(varis[f'Rx{loc}{t}'])
    for loc in range(1, n_components + 1):
        if not (loc == n_components and t == n_timesteps):
            varis_elim_order.append(varis[f'Clx{loc}{t}'])

# Initial Cx states
for loc in range(1, n_components + 1):
    varis_elim_order.append(varis[f'Cx{loc}0'])

# Cx temporal chain
for loc in range(1, n_components + 1):
    for t in range(1, n_timesteps + 1):
        varis_elim_order.append(varis[f'Cx{loc}{t}'])

print(varis_elim_order)
print(cpms)

marg_CrackLocationInterest = inference.variable_elim(cpms=cpms, var_elim=varis_elim_order, prod=True)
print(marg_CrackLocationInterest)


[<Variable representing Wx11['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x1e60eb6f3f0>, <Variable representing Lx11['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x1e60eb6e580>, <Variable representing Tx11['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x1e60eb6e5f0>, <Variable representing Wx21['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x1e60eb6e970>, <Variable representing Lx21['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x1e60eb6e9e0>, <Variable representing Tx21['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x1e60eb6ea50>, <Variable representing Px11['False', 'True'] at 0x1e60eb6e660>, <Variable representing Px21['False', 'True'] at 0x1e60eb6def0>, <Variable representing Vx11['-0.5', '-0.25', '0.0', '0.25', '0.5'] at 0x1e60eb6f310>, <Variable representing Vx21['-0.5', '-0.25', '0.0', '0.25', '0.5'] at 0x1e60eb6e0b0>, <Variable representing Ux1['-0.5', '-0.25', '0.0', '0.25', '0.5'] at 0x1e60ecd9860>, <Variable representing Zx11['-0.5', '-0.25', '0.0', '0.25', '0.5'] at 0x1e60eb6e900>, <Varia

In [ ]:
from mbnpy import variable, cpm, inference
import numpy as np
import copy

# ─────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────
n_components = 2
n_timesteps  = 2

# Observations per time step.
# Time-step level (shared across all locations):
#   'wx': observed weather state index 0-5 ('0','0.2','0.4','0.6','0.8','1.0'), or None
#   'lx': observed loading state index 0-5, or None
# Location level:
#   'clx': True if a crack is detected  → conditions Ux (spatial update)
#   'px':  True if a repair is carried out → resets Cx damage to 0 via C_Cx
observations = {
    t: {
        'wx': None,
        'lx': None,
        'locations': {loc: {'clx': False, 'px': False}
                      for loc in range(1, n_components + 1)}
    }
    for t in range(1, n_timesteps + 1)
}

# ─────────────────────────────────────────────────────────────
# STATE SPACES
# ─────────────────────────────────────────────────────────────
states_damage = ['0','0.2','0.4','0.6','0.8','1.0']
states_bool   = ['False','True']
states_resist = ['0.4','0.6','0.8','1.0','1.2','1.4']
states_aux    = ['-0.5','-0.25','0.0','0.25','0.5']

# ─────────────────────────────────────────────────────────────
# SHARED CPM MATRICES
# ─────────────────────────────────────────────────────────────
C_Tx = np.array([
    [0,0,0],[1,1,0],[2,2,0],[3,3,0],[4,4,0],[5,5,0],
    [1,0,1],[2,1,1],[3,2,1],[4,3,1],[5,4,1],[5,5,1],
    [0,0,2],[3,1,2],[4,2,2],[5,3,2],[5,4,2],[5,5,2],
    [3,0,3],[4,1,3],[5,2,3],[5,3,3],[5,4,3],[5,5,3],
    [4,0,4],[5,1,4],[5,2,4],[5,3,4],[5,4,4],[5,5,4],
    [5,0,5],[2,1,5],[5,2,5],[5,3,5],[5,4,5],[5,5,5],
], dtype=int)

C_Cx = np.array([
    [0,0,0,0],[0,1,0,0],[1,0,1,0],[1,1,1,0],[2,0,2,0],[0,1,2,0],
    [3,0,3,0],[0,1,3,0],[4,0,4,0],[0,1,4,0],[5,0,5,0],[0,1,5,0],
    [1,0,0,1],[1,1,0,1],[2,0,1,1],[1,1,1,1],[3,0,2,1],[1,1,2,1],
    [4,0,3,1],[1,1,3,1],[5,0,4,1],[1,1,4,1],[5,0,5,1],[1,1,5,1],
    [2,0,0,2],[2,1,0,2],[3,0,1,2],[2,1,1,2],[4,0,2,2],[2,1,2,2],
    [5,0,3,2],[2,1,3,2],[5,0,4,2],[2,1,4,2],[5,0,5,2],[2,1,5,2],
    [3,0,0,3],[3,1,0,3],[4,0,1,3],[3,1,1,3],[5,0,2,3],[3,1,2,3],
    [5,0,3,3],[3,1,3,3],[5,0,4,3],[3,1,4,3],[5,0,5,3],[3,1,5,3],
    [4,0,0,4],[4,1,0,4],[5,0,1,4],[4,1,1,4],[5,0,2,4],[4,1,2,4],
    [5,0,3,4],[4,1,3,4],[5,0,4,4],[4,1,4,4],[5,0,5,4],[4,1,5,4],
    [5,0,0,5],[5,1,0,5],[5,0,1,5],[5,1,1,5],[5,0,2,5],[5,1,2,5],
    [5,0,3,5],[5,1,3,5],[5,0,4,5],[5,1,4,5],[5,0,5,5],[5,1,5,5],
], dtype=int)

C_Rx = np.array([[1,0],[1,1],[2,2],[3,3],[4,4]], dtype=int)

C_Clx = np.array([
    [0,0,0],[0,1,0],[1,2,0],[1,3,0],[1,4,0],[1,5,0],
    [0,0,1],[0,1,1],[0,2,1],[1,3,1],[1,4,1],[1,5,1],
    [0,0,2],[0,1,2],[0,2,2],[0,3,2],[1,4,2],[1,5,2],
    [0,0,3],[0,1,3],[0,2,3],[0,3,3],[0,4,3],[1,5,3],
    [0,0,4],[0,1,4],[0,2,4],[0,3,4],[0,4,4],[0,5,4],
    [0,0,5],[0,1,5],[0,2,5],[0,3,5],[0,4,5],[0,5,5],
], dtype=int)

C_Zx = np.array([
    [0,0,0],[0,1,0],[1,2,0],[2,3,0],[2,4,0],
    [0,0,1],[1,1,1],[2,2,1],[2,3,1],[3,4,1],
    [1,0,2],[2,1,2],[2,2,2],[3,3,2],[4,4,2],
    [2,0,3],[2,1,3],[3,2,3],[4,3,3],[4,4,3],
    [3,0,4],[3,1,4],[4,2,4],[4,3,4],[4,4,4],
], dtype=int)

# ─────────────────────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────────────────────
def ux_prior_update(ux_marginal_cpm):
    """Normalise the Ux posterior from VE to use as the prior for the next time step."""
    p = ux_marginal_cpm.p.flatten()
    return p / p.sum()

# ─────────────────────────────────────────────────────────────
# INITIALISE
# ─────────────────────────────────────────────────────────────
varis = {}
cpms  = {}
ux_prior = np.array([0.0, 0.006, 0.493, 0.493, 0.006])  # default Ux prior

# Cx at t=0: no damage at any location
for loc in range(1, n_components + 1):
    varis[f'Cx{loc}0'] = variable.Variable(f'Cx{loc}0', states_damage)
    cpms[f'Cx{loc}0'] = cpm.Cpm(
        variables=[varis[f'Cx{loc}0']],
        no_child=1,
        C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
        p=np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])
    )

# ─────────────────────────────────────────────────────────────
# SEQUENTIAL TIME STEP LOOP
# ─────────────────────────────────────────────────────────────
for t in range(1, n_timesteps + 1):
    print(f"\n{'='*50}\nTime step t={t}\n{'='*50}")

    obs_t = observations[t]

    # 1. Ux{t}: create with current prior (updated each step via ux_prior_update)
    varis[f'Ux{t}'] = variable.Variable(f'Ux{t}', states_aux)
    cpms[f'Ux{t}'] = cpm.Cpm(
        variables=[varis[f'Ux{t}']],
        no_child=1,
        C=np.array([[0],[1],[2],[3],[4]], dtype=int),
        p=ux_prior.copy()
    )

    # 2. Build variables and CPMs for all locations at time step t
    for loc in range(1, n_components + 1):
        varis[f'Wx{loc}{t}']  = variable.Variable(f'Wx{loc}{t}',  states_damage)
        varis[f'Lx{loc}{t}']  = variable.Variable(f'Lx{loc}{t}',  states_damage)
        varis[f'Tx{loc}{t}']  = variable.Variable(f'Tx{loc}{t}',  states_damage)
        varis[f'Px{loc}{t}']  = variable.Variable(f'Px{loc}{t}',  states_bool)
        varis[f'Cx{loc}{t}']  = variable.Variable(f'Cx{loc}{t}',  states_damage)
        varis[f'Clx{loc}{t}'] = variable.Variable(f'CLx{loc}{t}', states_bool)
        varis[f'Rx{loc}{t}']  = variable.Variable(f'Rx{loc}{t}',  states_resist)
        varis[f'Zx{loc}{t}']  = variable.Variable(f'Zx{loc}{t}',  states_aux)
        varis[f'Vx{loc}{t}']  = variable.Variable(f'Vx{loc}{t}',  states_aux)

        # Wx and Lx: use prior unless an observation is provided (shared across locations)
        wx_p = np.zeros(6); wx_p[obs_t['wx']] = 1.0 if obs_t['wx'] is not None else None
        wx_p = np.array([0.017, 0.435, 0.518, 0.03, 0.0, 0.0]) if obs_t['wx'] is None else wx_p
        lx_p = np.zeros(6); lx_p[obs_t['lx']] = 1.0 if obs_t['lx'] is not None else None
        lx_p = np.array([0.122, 0.677, 0.198, 0.002, 0.0, 0.0]) if obs_t['lx'] is None else lx_p

        cpms[f'Wx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Wx{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
            p=wx_p
        )
        cpms[f'Lx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Lx{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
            p=lx_p
        )
        cpms[f'Tx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Tx{loc}{t}'], varis[f'Wx{loc}{t}'], varis[f'Lx{loc}{t}']],
            no_child=1,
            C=C_Tx,
            p=np.ones(len(C_Tx))
        )
        # Px=True: repair carried out → Cx reset to 0 via C_Cx (deterministic)
        px_p = np.array([0.0, 1.0]) if obs_t['locations'][loc]['px'] else np.array([1.0, 0.0])
        cpms[f'Px{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Px{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1]], dtype=int),
            p=px_p
        )
        cpms[f'Vx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Vx{loc}{t}']],
            no_child=1,
            C=np.array([[0],[1],[2],[3],[4]], dtype=int),
            p=np.array([0.0, 0.006, 0.493, 0.493, 0.006])
        )
        cpms[f'Zx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Zx{loc}{t}'], varis[f'Vx{loc}{t}'], varis[f'Ux{t}']],
            no_child=1,
            C=C_Zx,
            p=np.ones(len(C_Zx))
        )
        cpms[f'Rx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Rx{loc}{t}'], varis[f'Zx{loc}{t}']],
            no_child=1,
            C=C_Rx,
            p=np.ones(len(C_Rx))
        )
        cpms[f'Cx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Cx{loc}{t}'], varis[f'Px{loc}{t}'], varis[f'Cx{loc}{t-1}'], varis[f'Tx{loc}{t}']],
            no_child=1,
            C=C_Cx,
            p=np.ones(len(C_Cx))
        )
        cpms[f'Clx{loc}{t}'] = cpm.Cpm(
            variables=[varis[f'Clx{loc}{t}'], varis[f'Cx{loc}{t}'], varis[f'Rx{loc}{t}']],
            no_child=1,
            C=C_Clx,
            p=np.ones(len(C_Clx))
        )

    # 3. Condition on evidence for Ux inference:
    #    - Clx=True (crack detected) at any location
    #    - Wx and Lx observations are already baked into their CPM p-vectors above
    clx_cnd_vars, clx_cnd_states = [], []
    for loc in range(1, n_components + 1):
        if obs_t['locations'][loc]['clx']:
            clx_cnd_vars.append(varis[f'Clx{loc}{t}'])
            clx_cnd_states.append(1)  # index 1 = 'True'

    if clx_cnd_vars:
        cpms_ux = inference.condition(cpms, cnd_vars=clx_cnd_vars, cnd_states=clx_cnd_states)
    else:
        cpms_ux = copy.deepcopy(cpms)

    # 4. Elimination order: all variables in the model so far EXCEPT Ux{t} (the query).
    #    Order: Wx/Lx/Tx → Px → Vx → Ux{t<current} → Zx → Rx → Clx, then Cx chain.
    varis_elim_ux = []
    for t2 in range(1, t + 1):
        for loc in range(1, n_components + 1):
            varis_elim_ux += [varis[f'Wx{loc}{t2}'], varis[f'Lx{loc}{t2}'], varis[f'Tx{loc}{t2}']]
        for loc in range(1, n_components + 1):
            varis_elim_ux.append(varis[f'Px{loc}{t2}'])
        for loc in range(1, n_components + 1):
            varis_elim_ux.append(varis[f'Vx{loc}{t2}'])
        if t2 < t:
            varis_elim_ux.append(varis[f'Ux{t2}'])  # previous Ux vars are eliminated
        for loc in range(1, n_components + 1):
            varis_elim_ux.append(varis[f'Zx{loc}{t2}'])
        for loc in range(1, n_components + 1):
            varis_elim_ux.append(varis[f'Rx{loc}{t2}'])
        for loc in range(1, n_components + 1):
            varis_elim_ux.append(varis[f'Clx{loc}{t2}'])
    for loc in range(1, n_components + 1):
        varis_elim_ux.append(varis[f'Cx{loc}0'])
    for loc in range(1, n_components + 1):
        for t2 in range(1, t + 1):
            varis_elim_ux.append(varis[f'Cx{loc}{t2}'])

    # 5. VE: query Ux{t}
    ux_marginal = inference.variable_elim(cpms=cpms_ux, var_elim=varis_elim_ux, prod=True)
    print(f"Ux{t} posterior:")
    print(ux_marginal)

    # 6. Update Ux prior for next time step
    ux_prior = ux_prior_update(ux_marginal)
    print(f"Ux prior for t={t+1}: {np.round(ux_prior, 4)}")
